## Setup Ollama

In [ ]:
import sys
import time

# 0. Install zstd (required by Ollama installer)
print("Installing zstd dependency...")
!sudo apt-get update && sudo apt-get install -y zstd

# 1. Install Ollama inside the Colab runtime
print("Installing Ollama CLI...")
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start the Ollama server in the background
print("Starting Ollama server...")
# Use nohup to run in background and redirect output to a log file
# The `&` sends the process to the background
!nohup ollama serve > ollama.log 2>&1 &

# Wait for the Ollama server to initialize
print("Waiting for Ollama server to start (10 seconds)...")
time.sleep(10)

# 3. Pull the specified model
print("Pulling llama3.2:1b model...")
# The ollama command should now be available
!ollama pull llama3.2:1b

# 4. Install the ollama python client and pydantic
print("Installing ollama and pydantic Python packages...")
!{sys.executable} -m pip install ollama pydantic

print("Ollama setup complete.")

Installing zstd dependency...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,845 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,119 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://securi

In [ ]:
!ollama list

NAME           ID              SIZE      MODIFIED      
llama3.2:1b    baf6a787fdff    1.3 GB    3 seconds ago    


## Emotion-aware chatbot Albert

### import necessary libraries

In [ ]:
import json
import math
import random
from dataclasses import dataclass, field
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display
import ipywidgets as widgets
from ollama import Client
from pydantic import BaseModel, Field

### Configuration

In [ ]:
MODEL_NAME = globals().get("MODEL_NAME", "llama3.2:1b") # Define the name of the LLM model to use
OLLAMA_HOST = "http://127.0.0.1:11434" # Define the host for the Ollama server
client = Client(host=OLLAMA_HOST) # Initialize the Ollama client with the specified host


# ---------------------------------------------------------------------
# STRUCTURED LLM OUTPUT
# ---------------------------------------------------------------------

# Define a Pydantic model for structured emotion analysis output from the LLM
class EmotionAnalysis(BaseModel):
    """Text-based emotional estimate returned by the local LLM."""

    emotion: Literal[
        "happy",
        "sad",
        "angry",
        "fearful",
        "frustrated",
        "neutral",
    ] # The detected emotion, restricted to a set of predefined labels

    confidence: float = Field(ge=0.0, le=1.0) # Confidence score of the emotion detection (0.0 to 1.0)
    valence: float = Field(ge=-1.0, le=1.0) # Emotional valence (negativity to positivity, -1.0 to 1.0)
    arousal: float = Field(ge=-1.0, le=1.0) # Emotional arousal (calmness to activation, -1.0 to 1.0)
    explanation: str # A short explanation for the emotion analysis

### Robot emotion state machine/ emotion engine

In [ ]:
# ---------------------------------------------------------------------
# ROBOT EMOTION ENGINE
# ---------------------------------------------------------------------

class RobotEmotionEngine:
    """
    Maintains Albert's simulated internal mood on paired emotion axes.

    Important:
    These values simulate an artificial personality. They are not evidence
    that the robot genuinely experiences emotions.
    """

    def __init__(self):
        # Initialize various emotion levels for the robot
        self.emotions = {
            "happy": 50,
            "sad": 50,
            "welcome": 50,
            "distant": 50,
            "friend": 50,
            "strange": 50,
            "curious": 50,
            "frustrated": 50,
            "fresh": 80,
            "tired": 20,
        }

        # Define pairs of opposite emotions
        self.opposites = {
            "happy": "sad",
            "sad": "happy",
            "welcome": "distant",
            "distant": "welcome",
            "friend": "strange",
            "strange": "friend",
            "curious": "frustrated",
            "frustrated": "curious",
            "fresh": "tired",
            "tired": "fresh",
        }

        # Map emotions to angular positions on an emotion axis (in degrees)
        self.emotion_axis = {
            "happy": 112,
            "welcome": 22,
            "friend": 67,
            "curious": 157,
            "sad": 292,
            "distant": 202,
            "strange": 247,
            "frustrated": 337,
        }

        # Initialize polar state variables (radius and angle)
        self.state_r = 0.0
        self.state_theta = 112.0 # Default angle
        self.active_mood = "NEUTRAL" # Current active mood
        self.update() # Initial update of the emotional state

    def change(self, emotion: str, value: float) -> None:
        # Change the value of a specific emotion
        if emotion not in self.emotions:
            raise KeyError(f"Unknown robot emotion: {emotion}")

        value = float(np.clip(value, 0, 100)) # Clip value to be between 0 and 100
        self.emotions[emotion] = value # Set the emotion value

        opposite = self.opposites.get(emotion) # Get the opposite emotion
        if opposite is not None:
            self.emotions[opposite] = 100.0 - value # Update the opposite emotion value

        self.update() # Recalculate the overall mood

    def nudge(self, emotion: str, amount: float) -> None:
        # Adjust an emotion by a small amount
        self.change(emotion, self.emotions[emotion] + amount)

    def update(self) -> None:
        # Update the robot's overall emotional state (polar coordinates and active mood)
        x_sum = 0.0
        y_sum = 0.0

        # Calculate sum of x and y components for each emotion vector
        for emotion, angle in self.emotion_axis.items():
            radius = self.emotions[emotion] - 50.0 # Normalize emotion value around 50
            radians = math.radians(angle) # Convert angle to radians
            x_sum += radius * math.cos(radians)
            y_sum += radius * math.sin(radians)

        self.state_r = math.hypot(x_sum, y_sum) # Calculate resultant radius (magnitude)

        if self.state_r > 1e-3: # If there's a significant emotional state
            theta = math.degrees(math.atan2(y_sum, x_sum)) # Calculate resultant angle
            self.state_theta = theta % 360 # Normalize angle to 0-360 degrees
        else:
            self.state_theta = 112.0 # Default angle if no strong emotion

        if self.state_r < 10.0: # If the emotional state is weak, set to neutral
            self.active_mood = "NEUTRAL"
            return

        def angular_distance(a: float, b: float) -> float:
            # Helper function to calculate shortest angular distance
            delta = abs(a - b)
            return min(delta, 360.0 - delta)

        # Determine the active mood based on the closest emotion axis
        self.active_mood = min(
            self.emotion_axis,
            key=lambda emotion: angular_distance(
                self.state_theta,
                self.emotion_axis[emotion],
            ),
        ).upper()

### Human Emotion State Model

In [ ]:
class HumanEmotionEngine:
    """
    Stores Albert's current estimate of the human's expressed emotion.

    This is an inference from text, not a measurement of the user's actual
    internal emotional state.
    """

    def __init__(self):
        # Initialize emotional state variables for the human
        self.emotions = {
            "happy": 50,
            "sad": 50,
            "friendly": 50,
            "distant": 50,
        }

        # Define pairs of opposite emotions
        self.opposites = {
            "happy": "sad",
            "sad": "happy",
            "friendly": "distant",
            "distant": "friendly",
        }

        # Map emotions to angular positions on a polar axis
        self.emotion_axis = {
            "happy": 45,
            "friendly": 90,
            "sad": 225,
            "distant": 270,
        }

        # Initialize polar state (magnitude and angle)
        self.state_r = 0.0
        self.state_theta = 45.0
        self.active_mood = "NEUTRAL" # Current active mood
        self.update() # Initial update of the emotional state

    def change(self, emotion: str, value: float) -> None:
        # Change the value of a specific emotion
        if emotion not in self.emotions:
            raise KeyError(f"Unknown human-emotion dimension: {emotion}")

        value = float(np.clip(value, 0, 100)) # Clip value to be between 0 and 100
        self.emotions[emotion] = value # Set the emotion value

        opposite = self.opposites[emotion] # Get the opposite emotion
        self.emotions[opposite] = 100.0 - value # Update the opposite emotion value
        self.update() # Recalculate the overall mood

    def update(self) -> None:
        # Update the human's overall emotional state (polar coordinates and active mood)
        x_sum = 0.0
        y_sum = 0.0

        # Calculate sum of x and y components for each emotion vector
        for emotion, angle in self.emotion_axis.items():
            radius = self.emotions[emotion] - 50.0 # Normalize emotion value around 50
            radians = math.radians(angle) # Convert angle to radians
            x_sum += radius * math.cos(radians)
            y_sum += radius * math.sin(radians)

        self.state_r = math.hypot(x_sum, y_sum) # Calculate resultant radius (magnitude)

        if self.state_r > 1e-3: # If there's a significant emotional state
            theta = math.degrees(math.atan2(y_sum, x_sum)) # Calculate resultant angle
            self.state_theta = theta % 360 # Normalize angle to 0-360 degrees
        else:
            self.state_theta = 45.0 # Default angle if no strong emotion

        if self.state_r < 10.0: # If the emotional state is weak, set to neutral
            self.active_mood = "NEUTRAL"
            return

        def angular_distance(a: float, b: float) -> float:
            # Helper function to calculate shortest angular distance
            delta = abs(a - b)
            return min(delta, 360.0 - delta)

        # Determine the active mood based on the closest emotion axis
        self.active_mood = min(
            self.emotion_axis,
            key=lambda emotion: angular_distance(
                self.state_theta,
                self.emotion_axis[emotion],
            ),
        ).upper()

### Setting up Memory

In [ ]:
# ---------------------------------------------------------------------
# MEMORY
# ---------------------------------------------------------------------

@dataclass
class ConversationMemory:
    """Manages the conversation history and known facts about the user."""

    # Stores known facts about the user, initialized with None values
    known_facts: dict = field(
        default_factory=lambda: {
            "name": None,
            "age": None,
            "favorite_food": None,
            "favorite_book": None,
            "favorite_subject": None,
            "favorite_song": None,
            "favorite_color": None,
        }
    )
    # List of past conversation turns, each a dictionary with 'role' and 'content'
    history: list[dict[str, str]] = field(default_factory=list)
    # Counter for the number of conversation turns
    turn_count: int = 0

    def add_turn(self, user_text: str, assistant_text: str) -> None:
        """Adds a user and assistant turn to the conversation history."""
        self.history.extend(
            [
                {"role": "user", "content": user_text},
                {"role": "assistant", "content": assistant_text},
            ]
        )

        # Keep the prompt compact for a small local model.
        # Only retain the last 12 turns to manage memory usage and context window
        self.history = self.history[-12:]
        self.turn_count += 1

### Defining LLM functions

In [ ]:
# ---------------------------------------------------------------------
# LLM FUNCTIONS
# ---------------------------------------------------------------------

def analyse_emotion(
    user_text: str,
    memory: ConversationMemory,
) -> EmotionAnalysis:
    """
    Uses the local LLM as a zero-shot, text-only emotion classifier.
    """

    # Get the last 6 turns of conversation history for context
    recent_context = memory.history[-6:]

    # Construct the prompt for the LLM to classify emotion
    prompt = f"""
Classify the emotion expressed in the latest user message.

Choose exactly one label:
happy, sad, angry, fearful, frustrated, neutral.

Return:
- emotion
- confidence from 0.0 to 1.0
- valence from -1.0 (very negative) to +1.0 (very positive)
- arousal from -1.0 (very calm) to +1.0 (very activated)
- one short explanation

Rules:
- Infer only what is supported by the text and recent context.
- Do not diagnose the user.
- If emotional evidence is weak or ambiguous, return neutral with low confidence.
- Treat sarcasm cautiously.

Recent context:
{json.dumps(recent_context, ensure_ascii=False)}

Latest message:
{user_text}
"""

    try:
        # Call the Ollama client to chat with the model
        response = client.chat(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a conservative text-emotion classifier. "
                        "Always follow the supplied JSON schema."
                    ),
                },
                {"role": "user", "content": prompt},
            ],
            format=EmotionAnalysis.model_json_schema(), # Ensure output adheres to EmotionAnalysis schema
            options={
                "temperature": 0.0, # Low temperature for consistent classification
                "num_predict": 180, # Max tokens to predict
            },
            keep_alive="30m", # Keep the model loaded for 30 minutes
        )

        # Validate and return the structured emotion analysis
        return EmotionAnalysis.model_validate_json(response.message.content)

    except Exception as error:
        # Fallback to neutral emotion if LLM call fails
        print(f"Emotion-analysis fallback used: {error}")
        return EmotionAnalysis(
            emotion="neutral",
            confidence=0.0,
            valence=0.0,
            arousal=0.0,
            explanation="The local classifier did not return a valid result.",
        )


def generate_reply(
    user_text: str,
    analysis: EmotionAnalysis,
    robot: RobotEmotionEngine,
    memory: ConversationMemory,
) -> str:
    """Generates a reply from Albert, the robot, based on user input and emotional state."""

    # Filter out unknown facts (None values)
    known_facts = {
        key: value
        for key, value in memory.known_facts.items()
        if value is not None
    }

    # Construct the system prompt for Albert's persona and context
    system_prompt = f"""
You are Albert, a friendly demonstration robot in a beginner AI-for-Robotics course.

The user's text was estimated as:
- emotion: {analysis.emotion}
- model-reported confidence: {analysis.confidence:.2f}
- valence: {analysis.valence:.2f}
- arousal: {analysis.arousal:.2f}

Albert's simulated internal mood is: {robot.active_mood}

Known user facts:
{json.dumps(known_facts, ensure_ascii=False)}

Response rules:
- Reply naturally in at most three short sentences.
- Adapt the tone cautiously to the estimated emotion.
- Do not assert that you know exactly how the user feels.
- Do not repeatedly name the emotion label.
- Ask at most one relevant follow-up question.
- Do not claim that Albert truly experiences emotions.
- For dangerous or medical situations, recommend contacting a qualified
  person or emergency service instead of pretending to be a professional.
"""

    # Prepare messages for the LLM, including system prompt and recent history
    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(memory.history[-8:]) # Include last 8 turns of conversation
    messages.append({"role": "user", "content": user_text})

    try:
        # Call the Ollama client to generate a reply
        response = client.chat(
            model=MODEL_NAME,
            messages=messages,
            options={
                "temperature": 0.55, # Moderate temperature for creative but controlled replies
                "num_predict": 140, # Max tokens for the reply
            },
            keep_alive="30m", # Keep the model loaded for 30 minutes
        )
        return response.message.content.strip()

    except Exception as error:
        # Fallback message if LLM call fails
        return f"I could not reach my local language model. ({error})"

### State Mapping

In [ ]:
# ---------------------------------------------------------------------
# STATE-MAPPING POLICY
# ---------------------------------------------------------------------

def apply_analysis_to_human_engine(
    analysis: EmotionAnalysis,
    human: HumanEmotionEngine,
) -> None:
    """
    Maps the categorical LLM output and continuous valence/arousal values
    onto the human polar state model.
    """

    # Use continuous values as the primary mapping to happiness and calmness.
    happiness = 50.0 + 50.0 * analysis.valence
    calmness = 50.0 - 50.0 * analysis.arousal

    # For low-confidence estimates, keep the emotion closer to neutral.
    weight = analysis.confidence
    happiness = 50.0 + weight * (happiness - 50.0)
    calmness = 50.0 + weight * (calmness - 50.0)

    # Apply the calculated happiness and calmness to the human emotion engine.
    human.change("happy", happiness)
    # Map calmness to 'friendly' as a social state
    human.change("friendly", calmness)


def update_robot_policy(
    analysis: EmotionAnalysis,
    robot: RobotEmotionEngine,
) -> None:
    """
    Handcrafted policy that changes Albert's simulated mood.

    The robot does not simply copy the human's emotion. For assistive
    interaction, sadness or fear should generally increase Albert's
    friendliness and welcoming behaviour.
    """

    # If confidence is low, Albert becomes curious.
    if analysis.confidence < 0.30:
        robot.change("curious", 62)
        return

    # Adjust robot's emotions based on the detected human emotion.
    if analysis.emotion == "happy":
        robot.change("happy", 80)
        robot.change("welcome", 78)
        robot.change("friend", 72)

    elif analysis.emotion == "sad":
        robot.change("welcome", 88)
        robot.change("friend", 82)
        robot.change("curious", 62)

    elif analysis.emotion == "fearful":
        robot.change("welcome", 92)
        robot.change("friend", 85)
        robot.change("curious", 58)

    elif analysis.emotion == "angry":
        robot.change("welcome", 68)
        robot.change("friend", 70)
        robot.change("frustrated", 54)

    elif analysis.emotion == "frustrated":
        robot.change("curious", 78)
        robot.change("welcome", 74)
        robot.change("friend", 72)

    else:
        # Default behavior for other emotions or unclear states.
        robot.change("curious", 62)
        robot.change("welcome", 60)


def drain_battery(robot: RobotEmotionEngine) -> None:
    """Simulates a small battery drain after each conversation turn."""
    # Reduce the 'fresh' emotion, ensuring it doesn't go below 10.
    robot.change("fresh", max(10, robot.emotions["fresh"] - 2))

### Setting up for visualizing the emotion states of both robot & human live

In [ ]:
# ---------------------------------------------------------------------
# LIVE VISUALISATION
# ---------------------------------------------------------------------

def _plot_engine(
    axis, # The matplotlib axis object to plot on
    title: str, # Title for the plot
    engine, # The emotion engine (RobotEmotionEngine or HumanEmotionEngine)
    vector_label: str, # Label for the resultant emotion vector
) -> None:
    """Helper function to plot the emotional state of a single engine on a polar axis."""
    axis.set_title(title, pad=22, fontweight="bold")

    # Plot individual emotion axes and their current values
    for emotion, angle in engine.emotion_axis.items():
        theta = math.radians(angle) # Convert angle to radians
        value = engine.emotions[emotion] # Get the current emotion value

        axis.plot([theta, theta], [0, 100], linestyle=":", alpha=0.45) # Draw emotion axis line
        axis.scatter([theta], [value], s=70, zorder=3) # Plot emotion value as a scatter point
        axis.text(
            theta,
            111,
            f"{emotion}\\n{value:.0f}", # Display emotion name and value
            ha="center",
            va="center",
            fontsize=8,
        )

    # Calculate and plot the resultant emotion vector
    result_radius = float(np.clip(engine.state_r, 0, 100)) # Clip resultant radius to bounds
    result_theta = math.radians(engine.state_theta) # Convert resultant angle to radians

    axis.annotate(
        "",
        xy=(result_theta, result_radius), # End point of the arrow
        xytext=(0, 0), # Start point of the arrow (origin)
        arrowprops={
            "arrowstyle": "-|>",
            "linewidth": 3,
        },
    )

    axis.scatter(
        [result_theta],
        [result_radius],
        marker="*",
        s=150,
        zorder=5,
        label=vector_label,
    )

    # Set plot limits and hide tick labels for cleanliness
    axis.set_ylim(0, 100)
    axis.set_xticklabels([])
    axis.set_yticklabels([])
    axis.grid(alpha=0.3)


def show_live_visualisation(
    robot: RobotEmotionEngine, # The robot's emotion engine
    human: HumanEmotionEngine, # The human's emotion engine
    analysis: EmotionAnalysis | None, # The latest emotion analysis from the LLM
    memory: ConversationMemory, # The conversation memory
) -> None:
    """Displays a live visualization of robot and human emotional states."""
    clear_output(wait=True) # Clear previous output to update the plot

    figure = plt.figure(figsize=(13, 5.8)) # Create a new figure

    # Create two polar subplots for robot and human emotions
    robot_axis = figure.add_subplot(1, 2, 1, projection="polar")
    human_axis = figure.add_subplot(1, 2, 2, projection="polar")

    # Plot robot's emotional state
    _plot_engine(
        robot_axis,
        "Albert's Simulated Internal Emotion State",
        robot,
        "Robot mood vector",
    )

    # Plot human's estimated emotional state
    _plot_engine(
        human_axis,
        "Albert's Estimate of the Human's Expressed Emotion",
        human,
        "Human-state vector",
    )

    # Prepare status text based on the latest emotion analysis
    if analysis is None:
        status = "No user message analysed yet."
    else:
        status = (
            f"Text estimate: {analysis.emotion.upper()} | "
            f"model-reported confidence: {analysis.confidence:.2f} | "
            f"valence: {analysis.valence:.2f} | "
            f"arousal: {analysis.arousal:.2f}"
        )

    # Add overall status text below the plots
    figure.text(
        0.5,
        0.03,
        (
            f"Robot mood: {robot.active_mood} | "
            f"Human-state estimate: {human.active_mood} | "
            f"Turns: {memory.turn_count}\\n{status}"
        ),
        ha="center",
        fontsize=10,
    )

    figure.tight_layout(rect=[0, 0.09, 1, 1]) # Adjust layout to prevent overlap
    plt.show() # Display the plot

### Setting up the interactive UI

In [ ]:
# ---------------------------------------------------------------------
# INTERACTIVE COLAB CHAT UI
# ---------------------------------------------------------------------

# Initialize emotion engines and conversation memory
robot_engine = RobotEmotionEngine()
human_engine = HumanEmotionEngine()
memory = ConversationMemory()
last_analysis = None # Stores the result of the last emotion analysis

# Define UI widgets
chat_log = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #aaa", # Styling for the chat log box
        height="300px",
        overflow_y="auto",
        padding="8px",
    )
)

visual_output = widgets.Output() # Output area for the live visualization

message_box = widgets.Textarea(
    placeholder="Type a message to Albert...", # Placeholder text for input
    description="You:", # Label for the input box
    layout=widgets.Layout(width="78%", height="72px"), # Layout dimensions
)

send_button = widgets.Button(
    description="Send", # Text on the button
    button_style="primary", # Styling for the button
    icon="paper-plane", # Icon displayed on the button
)

reset_button = widgets.Button(
    description="Reset", # Text on the reset button
    icon="refresh", # Icon for the reset button
)

status_label = widgets.HTML(
    value=f"<b>Local model:</b> {MODEL_NAME}" # Display current model status
)


def refresh_visualisation() -> None:
    """Updates the live emotional state visualization."""
    with visual_output:
        show_live_visualisation(
            robot_engine,
            human_engine,
            last_analysis,
            memory,
        )


def submit_message(_=None) -> None:
    """Handles user message submission: analyzes, generates reply, and updates UI."""
    global last_analysis # Declare last_analysis as global to modify it

    user_text = message_box.value.strip() # Get and clean user input
    if not user_text:
        return # Do nothing if message is empty

    # Disable controls and update status while processing
    send_button.disabled = True
    message_box.disabled = True
    status_label.value = "<b>Status:</b> Analysing emotion and generating reply..."

    try:
        # Process user message: analyze emotion, update engines, generate reply
        analysis = analyse_emotion(user_text, memory) # Analyze user's emotion
        last_analysis = analysis # Store the latest analysis

        apply_analysis_to_human_engine(analysis, human_engine) # Update human emotion engine
        update_robot_policy(analysis, robot_engine) # Update robot's emotion policy
        drain_battery(robot_engine) # Simulate battery drain for robot

        reply = generate_reply(
            user_text=user_text,
            analysis=analysis,
            robot=robot_engine,
            memory=memory,
        ) # Generate Albert's reply

        memory.add_turn(user_text, reply) # Add conversation turn to memory

        # Display conversation and analysis in chat log
        with chat_log:
            print(f"You: {user_text}")
            print(
                "Text-emotion estimate: "
                f"{analysis.emotion.upper()} "
                f"(model-reported confidence={analysis.confidence:.2f}, "
                f"valence={analysis.valence:.2f}, "
                f"arousal={analysis.arousal:.2f})"
            )
            print(f"Albert: {reply}")
            print("-" * 72)

        message_box.value = "" # Clear message input box
        refresh_visualisation() # Update the live visualization
        status_label.value = f"<b>Local model:</b> {MODEL_NAME}" # Restore status label

    finally:
        # Re-enable controls after processing, regardless of success or failure
        send_button.disabled = False
        message_box.disabled = False


def reset_demo(_=None) -> None:
    """Resets the chat demo to its initial state."""
    global robot_engine, human_engine, memory, last_analysis # Declare globals to reset them

    # Re-initialize all state variables
    robot_engine = RobotEmotionEngine()
    human_engine = HumanEmotionEngine()
    memory = ConversationMemory()
    last_analysis = None

    chat_log.clear_output() # Clear the chat log display
    refresh_visualisation() # Refresh visualization to initial state
    status_label.value = f"<b>Local model:</b> {MODEL_NAME} — conversation reset." # Update status


# Attach event handlers to buttons
send_button.on_click(submit_message)
reset_button.on_click(reset_demo)

# Display the UI widgets in the notebook
controls = widgets.HBox([send_button, reset_button]) # Group send and reset buttons
display(status_label, chat_log, message_box, controls, visual_output)

# Initial messages in the chat log
with chat_log:
    print("Albert: Hello! I am Albert, a local emotion-aware demonstration robot.")
    print("Type a message below. The emotion estimate is based only on your text.")
    print("-" * 72)

refresh_visualisation() # Display initial visualization

HTML(value='<b>Local model:</b> llama3.2:1b')

Output(layout=Layout(border='1px solid #aaa', height='300px', overflow_y='auto', padding='8px'))

Textarea(value='', description='You:', layout=Layout(height='72px', width='78%'), placeholder='Type a message …

Output()